# Week 6 build · BPE from scratch, one ablation, one scaling line

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-06.ipynb)

**~3 hours · T4 recommended for parts C–E · run top to bottom.** With `SMOKE = True` (next cell) everything runs on a CPU in a few minutes at toy size; set it to `False` on a T4 for the real numbers.

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running. The notebook refuses empty predictions. This week that rule is the deliverable: an ablation without a prior prediction is a demo, not an experiment (canvas steps 7–8).

| Part | Topic | Time |
|---|---|---|
| A | A BPE trainer from scratch | 30 min |
| B | GPT-2's real tokenizer (optional) | 15 min |
| C | nanoGPT-mini with four switches; the untouched model, twice | 25 min |
| D | The ablation, two random starts each | 60 min |
| E | Three sizes: a scaling line and 6ND | 40 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Setup · the SMOKE switch

`SMOKE = True` shrinks the data, the model and the step count so every cell runs on a CPU in minutes. It proves the pipeline; the numbers are toys. `SMOKE = False` is the real run (96-wide, 3 layers, 1,500 steps, the config the lesson-5 visual's reference numbers came from).

In [ ]:
SMOKE = True   # True: tiny model, ~50 steps, CPU in minutes. False: the real run (T4, ~15 min of training)
import os, sys, math, time, urllib.request
import torch, torch.nn as nn, torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STEPS, BS = (50, 16) if SMOKE else (1500, 32)
BASE = dict(n_embd=32 if SMOKE else 96, n_head=4, n_layer=2 if SMOKE else 3, block=32 if SMOKE else 64,
            use_pos=True, use_ln=True, causal=True)
print(f"device={DEVICE}  mode={'SMOKE' if SMOKE else 'REAL'}  steps={STEPS}  batch={BS}  base config={BASE}")
if not SMOKE and DEVICE == "cpu":
    print("No GPU: Runtime -> Change runtime type -> T4 GPU, or set SMOKE = True")

---
## Part A · A BPE trainer from scratch (30 min)

**Words used in this part.** *byte*: a number 0–255; text is stored as bytes (one per English letter, three per Korean syllable). *merge rule*: glue this pair of tokens into a new token; applied in learned order. *vocabulary*: 256 bytes plus one token per merge. *BPE*: repeat "count adjacent pairs, merge the most frequent" until the vocabulary is the size you want.

In [ ]:
#@title 🔮 Predict A1: on 'aaabdaaabac', which pair merges first, and how many tokens remain after 3 merges?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
from collections import Counter

def pair_counts(ids):
    return Counter(zip(ids, ids[1:]))

def apply_merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

def train_bpe(text, n_merges, verbose=False):
    ids, merges, vocab = list(text.encode("utf-8")), [], {i: bytes([i]) for i in range(256)}
    for k in range(n_merges):
        counts = pair_counts(ids)
        if not counts or max(counts.values()) < 2: break          # nothing repeats: stop early
        pair = counts.most_common(1)[0][0]
        new_id = 256 + k
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
        merges.append((pair, new_id)); ids = apply_merge(ids, pair, new_id)
        if verbose: print(f"merge {k+1}: {vocab[pair[0]]!r} + {vocab[pair[1]]!r} -> id {new_id}   tokens left: {len(ids)}")
    return merges, vocab, ids

In [ ]:
merges, vocab, ids = train_bpe("aaabdaaabac", 3, verbose=True)
print("final tokens:", [vocab[i] for i in ids], "->", len(ids), "tokens from 11 bytes")

In [ ]:
#@title ✍️ A1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A1", what_happened, surprise)

**Now real text.** Tiny Shakespeare is 1.1 MB; we train on the first 50 KB (8 KB in SMOKE mode). Each merge re-scans the whole text, which is why real tokenizer trainers are written in Rust.

In [ ]:
#@title 🔮 Predict A2: after 300 merges on Shakespeare (60 in SMOKE), how many bytes per token? What will the very first merge be?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A2", prediction, confidence)

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    text = urllib.request.urlopen(URL, timeout=30).read().decode("utf-8")
    print(f"tiny Shakespeare: {len(text):,} characters")
except Exception as e:
    print("download failed:", e, "\n-> using a small built-in stand-in text; your numbers will differ")
    text = "First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\n" * 400

In [ ]:
sample = text[:8_000] if SMOKE else text[:50_000]
N_MERGES = 60 if SMOKE else 300
t0 = time.time(); merges, vocab, ids = train_bpe(sample, N_MERGES)
n_bytes = len(sample.encode("utf-8"))
print(f"{len(merges)} merges in {time.time()-t0:.1f}s: {n_bytes:,} bytes -> {len(ids):,} tokens = {n_bytes/len(ids):.2f} bytes per token")
print("first five merges:", [vocab[new] for _, new in merges[:5]])
print("longest tokens:  ", sorted((vocab[new] for _, new in merges), key=len)[-6:])

def encode(s, merges=merges):
    ids = list(s.encode("utf-8"))
    for pair, new_id in merges:              # apply the rules in the order they were learned
        ids = apply_merge(ids, pair, new_id)
    return ids

for s in ["the king", " the king", "strawberry", "20240928", "안녕하세요"]:
    print(f"{s!r:16} {len(s.encode('utf-8')):3d} bytes -> {len(encode(s)):3d} tokens:", [vocab[i] for i in encode(s)])

In [ ]:
#@title ✍️ A2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A2", what_happened, surprise)

Notice three things: the first merges are the most common letter pairs *including the space* (`e ` or ` t`), so tokens straddle word boundaries unless a pre-tokenizer splits words first (GPT-2 does); `strawberry` stays in pieces; and the Korean line is three tokens per syllable, one per byte, because nothing Korean was ever merged. That is lesson 2's whole argument in one cell.

---
## Part B · GPT-2's real tokenizer (15 min, optional)

**Words used in this part.** *tokenizer artefact*: a model behaviour caused by how the text was cut, not by the model. `tiktoken` is OpenAI's tokenizer library; on Colab the cell installs it, elsewhere it skips if missing.

In [ ]:
#@title 🔮 Predict B1: GPT-2 tokens for 'Hello, welcome to Seoul.' and for '안녕하세요, 서울에 오신 것을 환영합니다.' (5 + 14 characters of Korean). Guess both counts.
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
try:
    import tiktoken
except ImportError:
    tiktoken = None
    if "COLAB_RELEASE_TAG" in os.environ:                       # auto-install only on Colab
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=False)
        try: import tiktoken
        except ImportError: pass
if tiktoken is None:
    print("tiktoken not installed: skipped. `pip install tiktoken` and re-run; lesson 2's table has the counts.")
else:
    for name in ["gpt2", "cl100k_base"]:
        enc = tiktoken.get_encoding(name)
        for s in ["Hello, welcome to Seoul.", "안녕하세요, 서울에 오신 것을 환영합니다.", "strawberry", " strawberry", "20240928"]:
            ids = enc.encode(s)
            pieces = [enc.decode_single_token_bytes(i).decode("utf-8", "replace") for i in ids]
            print(f"{name:12} {len(ids):3d} tokens  {s!r:44} {pieces[:8]}")

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

---
## Part C · nanoGPT-mini with four switches (25 min)

**Words used in this part.** *causal mask*: position t may attend only to positions ≤ t (`causal=True`). *position encoding*: a learned vector per position added to the token vector, the only thing that tells attention about order (`use_pos`). *LayerNorm*: per-token normalisation before attention and MLP (`use_ln`). *heads*: attention split into `n_head` parallel lookups. *val loss*: cross-entropy on held-back text; random guessing gives ln(vocab).

This is the week-5 model with the four components made optional. Read the three cells; the switches are the lines with `if`.

In [ ]:
class Attention(nn.Module):
    def __init__(s, d, n_head, block, causal=True):
        super().__init__()
        s.n_head, s.causal = n_head, causal
        s.qkv, s.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        s.register_buffer("mask", torch.tril(torch.ones(block, block)))
    def forward(s, x):
        B, T, C = x.shape
        q, k, v = s.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, s.n_head, C // s.n_head).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        if s.causal: att = att.masked_fill(s.mask[:T, :T] == 0, float("-inf"))   # <- the mask switch
        y = F.softmax(att, dim=-1) @ v
        return s.proj(y.transpose(1, 2).reshape(B, T, C))

In [ ]:
class Block(nn.Module):
    def __init__(s, cfg):
        super().__init__()
        d = cfg["n_embd"]
        s.ln1 = nn.LayerNorm(d) if cfg["use_ln"] else nn.Identity()      # <- the LayerNorm switch
        s.ln2 = nn.LayerNorm(d) if cfg["use_ln"] else nn.Identity()
        s.attn = Attention(d, cfg["n_head"], cfg["block"], cfg["causal"]) # <- the heads switch is n_head
        s.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(s, x):
        x = x + s.attn(s.ln1(x))
        return x + s.mlp(s.ln2(x))

In [ ]:
class GPT(nn.Module):
    def __init__(s, cfg):
        super().__init__()
        s.cfg = cfg
        s.tok = nn.Embedding(cfg["vocab"], cfg["n_embd"])
        s.pos = nn.Embedding(cfg["block"], cfg["n_embd"]) if cfg["use_pos"] else None   # <- the position switch
        s.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg["n_layer"])])
        s.ln_f = nn.LayerNorm(cfg["n_embd"]) if cfg["use_ln"] else nn.Identity()
        s.head = nn.Linear(cfg["n_embd"], cfg["vocab"], bias=False)
    def forward(s, idx, targets=None):
        x = s.tok(idx)
        if s.pos is not None: x = x + s.pos(torch.arange(idx.size(1), device=idx.device))
        logits = s.head(s.ln_f(s.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
if SMOKE: data = data[:20_000]
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
BASE["vocab"] = len(chars)
print(f"vocab {len(chars)}  train {len(train_data):,} chars  val {len(val_data):,} chars  "
      f"random-guess loss ln({len(chars)}) = {math.log(len(chars)):.2f}")

def get_batch(split, cfg):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - cfg["block"] - 1, (BS,))
    x = torch.stack([d[i:i + cfg["block"]] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + cfg["block"]] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def val_loss(model, cfg, n_batches=20):
    model.eval(); losses = [model(*get_batch("val", cfg))[1].item() for _ in range(n_batches)]; model.train()
    return sum(losses) / n_batches

In [ ]:
def n_params(model): return sum(p.numel() for p in model.parameters())

def train(cfg, seed, steps=None, lr=1e-3):
    steps = steps or STEPS
    torch.manual_seed(seed)                                   # the random start: init + batch order
    model = GPT(cfg).to(DEVICE); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    curve, t0 = [], time.time()
    for step in range(steps + 1):
        if step % max(steps // 10, 1) == 0: curve.append((step, val_loss(model, cfg)))
        if step == steps: break
        _, loss = model(*get_batch("train", cfg))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    print(f"  seed {seed}: {n_params(model):,} params, val loss {curve[-1][1]:.3f} after {steps} steps ({time.time()-t0:.0f}s)")
    return model, curve

In [ ]:
#@title 🔮 Predict C1: the untouched model, two random starts: final val loss? And how far apart will the two runs end (the seed spread)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
RUNS = {}
def run(name, cfg, seeds=(0, 1)):
    changed = {k: v for k, v in cfg.items() if BASE.get(k) != v}
    print(name, changed or "(untouched)")
    RUNS[name] = [train(cfg, s) for s in seeds]
    finals = [c[-1][1] for _, c in RUNS[name]]
    print(f"  mean {sum(finals)/len(finals):.3f}   seed spread {max(finals)-min(finals):.3f}")
run("baseline", BASE)

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

Keep that seed spread in view: it is the size of change the untouched model shows *for no reason at all*. An ablation whose effect is inside the spread has told you nothing yet; you would need more runs.

---
## Part D · The ablation (60 min)

**Words used in this part.** *ablation*: remove or replace one component, keep everything else fixed, measure the change in loss. *effect vs spread*: the gap between ablated and untouched means, compared with how much two untouched runs differ.

Pick **one** of the four switches. Write the direction *and* the rough size of the change you expect, and why, before running. (`no_mask` is the one lesson 5 said you can predict exactly; save it for D2 unless you cannot resist.)

In [ ]:
#@title 🔮 Predict D1: pick ONE component to remove, then predict the val loss change vs the untouched model (direction and size)
component = "no_pos"  #@param ["no_pos", "no_ln", "one_head", "no_mask"]
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", f"{component}: {prediction}", confidence)

In [ ]:
import matplotlib.pyplot as plt
ABLATIONS = {"no_pos": {"use_pos": False}, "no_ln": {"use_ln": False}, "one_head": {"n_head": 1}, "no_mask": {"causal": False}}
run(component, {**BASE, **ABLATIONS[component]})

def compare(*names):
    for name in names:
        for seed, (_, curve) in enumerate(RUNS[name]):
            plt.plot(*zip(*curve), "-" if name == "baseline" else "--", label=f"{name} seed {seed}")
    plt.xlabel("step"); plt.ylabel("val loss"); plt.legend(); plt.show()
    b = [c[-1][1] for _, c in RUNS["baseline"]]
    for name in names[1:]:
        f = [c[-1][1] for _, c in RUNS[name]]
        print(f"{name:9s} mean {sum(f)/2:.3f}  untouched {sum(b)/2:.3f}  effect {sum(f)/2 - sum(b)/2:+.3f}  "
              f"seed spread {max(f)-min(f):.3f} (untouched {max(b)-min(b):.3f})")
compare("baseline", component)

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

**Explain it in one sentence** before moving on: the component, the sign and size of the effect, and whether the effect is larger than the seed spread. If it is not, say what you would need (more runs, more steps) before claiming anything. In SMOKE mode, 50 steps is before the model has really used the component: effects can sit inside the spread or even point the wrong way. That is the honest result at toy size, and the reason the real run exists.

Now the second one. Default `no_mask`: lesson 5 says the loss should collapse toward zero because position t can read token t+1. In the real run it reaches about 0.04 by 1,500 steps; in SMOKE mode the copying trick has not been learned yet at 50 steps, so expect little. If you already ran `no_mask`, pick `no_pos` instead and predict how attention copes with no idea of order.

In [ ]:
#@title 🔮 Predict D2: the second ablation
component2 = "no_mask"  #@param ["no_mask", "no_pos", "no_ln", "one_head"]
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", f"{component2}: {prediction}", confidence)

In [ ]:
run(component2, {**BASE, **ABLATIONS[component2]})
compare("baseline", component, component2)
if "no_mask" in RUNS:
    print("no_mask: the last position in each window has no t+1 to copy, so the floor is about (untouched loss) / block "
          f"= {sum(c[-1][1] for _, c in RUNS['baseline'])/2/BASE['block']:.3f}, not zero."
          + ("  (SMOKE: the copy trick takes a few hundred steps to appear; the real run gets there.)" if SMOKE else ""))

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

---
## Part E · Three sizes, same tokens (40 min)

**Words used in this part.** *power law*: loss ∝ N^−α; each ×10 in N multiplies loss by the same factor. *log–log plot*: both axes in powers of ten; a power law is a straight line whose slope is −α. *6ND*: training FLOPs ≈ 6 × parameters × tokens.

Three models, small / medium / large, each trained on exactly the same number of tokens (`STEPS × BS × block`). Kaplan's slope at scale is −0.076; yours will be steeper, because tiny models are far from the floor.

In [ ]:
#@title 🔮 Predict E1: three sizes (about 10x params apart). Will val loss vs parameters be a straight line on log-log axes? What slope (negative number) do you expect?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
SIZES = ({"S": dict(n_embd=16, n_layer=1, n_head=2), "M": dict(n_embd=32, n_layer=2, n_head=4), "L": dict(n_embd=64, n_layer=2, n_head=4)}
         if SMOKE else
         {"S": dict(n_embd=32, n_layer=2, n_head=2), "M": dict(n_embd=64, n_layer=3, n_head=4), "L": dict(n_embd=128, n_layer=4, n_head=4)})
scale = {}
for name, over in SIZES.items():
    print(name, over)
    model, curve = train({**BASE, **over}, seed=0)
    scale[name] = (n_params(model), curve[-1][1])

In [ ]:
import numpy as np
Ns, Ls = zip(*scale.values())
slope, intercept = np.polyfit(np.log10(Ns), np.log10(Ls), 1)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for a, kind in zip(ax, ["linear", "log"]):
    a.plot(Ns, Ls, "o-"); a.set_xscale(kind); a.set_yscale(kind)
    a.set_xlabel("parameters N"); a.set_ylabel("val loss"); a.set_title(kind + " axes")
plt.tight_layout(); plt.show()
D = STEPS * BS * BASE["block"]
print(f"tokens per run D = {D:,}    fitted log-log slope = {slope:.3f}   (Kaplan at scale: -0.076)")
for name, (N, L) in scale.items():
    print(f"{name}: N = {N:>9,}   val loss {L:.3f}   tokens/param {D/N:8.1f}   6ND = {6*N*D:.2e} FLOPs")
print("Is the middle point on the line through the outer two? If not, which way does it bend, and why might that be?")

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

Three points is not a law; it is a sanity check that the *shape* Kaplan drew also appears in a model you built. The gap between your slope and −0.076 is itself informative: far from the floor, every parameter still buys a lot.

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `core-transformer` (the ablation rows) and `move-scale` (the scaling row), and mark the biggest surprise.
Then on the laptop:

```
python track.py done week-06/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 7
```

In [ ]:
reveal()